# 01 — Kiểm chứng processor, loader và PDP nền

**Trạng thái bàn giao: chưa chạy.** Tạo ngày 08/10/2026. Chạy sau khi tự cài [dependencies](../../docs/environment/README.md), chọn kernel Python 3.10 và cung cấp release native đã khóa. Không có lệnh cài package/API download trong notebook.

Mục tiêu: kiểm tra train JSON 5 task, separation full GT; resize/normalize/pad và bbox round-trip ở 640/800; loader/collation/shuffle; forward/backward, detection/query/DDL losses, optimizer; teacher/PPG/prototypes và Lightning resume qua các tests hiện có.

Input: `rpc_100-4x25_seed0_native_v1` + ảnh gốc `data/archive`. Output: `runs/modeling/processor_loader_pdp/<run_id>/` (config, gates, runtime, logs, JUnit XML, source snapshot). Không ghi vào release, raw data hoặc test lock. Không tích hợp synth 202 trong lượt này.

**Phạm vi smoke PDP:** ResNet18 ngẫu nhiên, encoder 1/decoder 2, prompt 4×2, ảnh 96; giữ 300 queries/256 dimensions và mapping 224 slots/225 outputs. Processor/loader 640/800 dùng ảnh thật, còn smoke nhỏ chỉ kiểm chứng đường tính toán. Nghiệm thu smoke không xác nhận pretrained ResNet50 đầy đủ, model forward/backward ở 640/800, ConvNeXt hay convergence. Các phần đó cần notebook/run kế tiếp.

Chạy lần lượt từ setup tới cell cuối. Nếu tắt smoke/regression/resume, kết quả phải PARTIAL. Lưu notebook sau khi chạy để giữ outputs; started_at/finished_at chỉ được tạo khi thực chạy.

## 1. Cấu hình, run ID và nhật ký

Sửa đường dẫn trước khi chạy setup. Resolution dùng `shortest_edge=longest_edge=R`: giữ tỷ lệ ảnh, cạnh dài tối đa R, padding theo batch. Ảnh gần vuông thường ra R×R; không kéo giãn ảnh chữ nhật. Batch chọn cố định 2 ảnh đầu mỗi task, workers=0 và không augmentation để kiểm tra geometry nền. Smoke/resume dùng CPU; phần GPU/Vast.ai do bạn cấu hình riêng.

In [ ]:
import os, sys, json, uuid, subprocess, hashlib, time, platform, gc
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from contextlib import contextmanager
from importlib import metadata

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pdp/main.py').exists())
for directory in [REPO, REPO/'pdp', REPO/'tests']:
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))
from autocheckout.io import load_json, save_json, md5_file

RAW = Path(os.getenv('AUTOCHECKOUT_DATA_ROOT', str(REPO/'data/archive'))).resolve()
RELEASE = Path(os.getenv('AUTOCHECKOUT_RELEASE_ROOT', str(REPO/'data/processed/rpc_100-4x25_seed0_native_v1'))).resolve()
CONFIG = {'seed': 0, 'resolutions': [640, 800], 'batch_size': 2,
          'smoke_resolution': 96, 'run_smoke': True,
          'run_baseline_regression': True, 'run_resume_regression': True,
          'workers': 0, 'device': 'cpu', 'augment': False,
          'raw_root': str(RAW), 'release_root': str(RELEASE),
          'profile': 'tiny_random_pdp_on_real_task1_batch'}
assert CONFIG['resolutions'] == [640, 800] and CONFIG['batch_size'] == 2
os.environ['USE_TF'] = '0'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('MPLBACKEND', 'Agg')

ARTIFACT = REPO/'runs/modeling/processor_loader_pdp'/str(uuid.uuid4())
ARTIFACT.mkdir(parents=True)
os.environ.setdefault('MPLCONFIGDIR', str(ARTIFACT/'matplotlib'))
now = lambda: datetime.now(ZoneInfo('Asia/Bangkok')).isoformat(timespec='seconds')
REQUIRED = ['environment', 'release_lock', 'task_gt_separation', 'processor_640',
            'processor_800', 'loader_640', 'loader_800', 'pdp_smoke',
            'baseline_regression', 'lightning_resume', 'sources_preserved']
RUN = {'run_id': ARTIFACT.name, 'purpose': 'Processor/loader and PDP foundation acceptance',
       'started_at': now(), 'finished_at': None, 'timezone': 'Asia/Bangkok',
       'status': 'RUNNING', 'python': sys.version, 'executable': sys.executable,
       'host': platform.node(), 'config': CONFIG,
       'git_commit': subprocess.check_output(['git','rev-parse','HEAD'], cwd=REPO, text=True).strip(),
       'git_dirty': bool(subprocess.check_output(['git','status','--porcelain'], cwd=REPO, text=True).strip()),
       'gates': {}, 'full_pdp_training_accepted': False,
       'full_model_640_800_forward_backward_accepted': False,
       'vast_gpu_accepted': False, 'convnext_accepted': False}

def persist():
    save_json(ARTIFACT/'run.json', RUN, indent=2)
    save_json(ARTIFACT/'gates.json', RUN['gates'], indent=2)

@contextmanager
def gate(name):
    record = {'status': 'RUNNING', 'started_at': now()}
    RUN['gates'][name] = record
    RUN['status'] = 'RUNNING'
    RUN['finished_at'] = None
    persist()
    try:
        yield record
    except BaseException as error:
        record.update(status='INTERRUPTED' if isinstance(error, KeyboardInterrupt) else 'FAIL',
                      error_type=type(error).__name__, error=str(error), finished_at=now())
        RUN.update(status=record['status'], finished_at=record['finished_at'])
        persist()
        raise
    else:
        record.update(status='PASS', finished_at=now())
        persist()
        print(name, 'PASS')

def skip(name):
    RUN['gates'][name] = {'status': 'SKIPPED', 'reason': 'Disabled in CONFIG', 'recorded_at': now()}
    persist()

persist()
save_json(ARTIFACT/'config.json', CONFIG, indent=2)
print('Run:', RUN['run_id'], 'Started:', RUN['started_at'], 'Artifacts:', ARTIFACT)

## 2. Gate môi trường và imports

Kiểm tra Python 3.10 và pins runtime/dev theo files requirements. Notebook không sửa/cài môi trường nếu thiếu gói; gate FAIL ghi rõ lỗi. Bản torch CPU/CUDA có suffix (`+cpu`, `+cu121`) được chấp nhận theo base version. CUDA availability chỉ được ghi nhận, không tự chứng nhận custom kernel/GPU.

In [ ]:
with gate('environment') as evidence:
    assert sys.version_info[:2] == (3, 10), 'Choose the project Python 3.10 kernel'
    expected = {'torch': '2.2.2', 'torchvision': '0.17.2'}
    for requirement_file in ['requirements.txt']:
        for line in (REPO/requirement_file).read_text().splitlines():
            line = line.split('#', 1)[0].strip()
            if line and '==' in line:
                package, version = line.split('==', 1)
                expected[package] = version
    installed = {}
    for package, version in expected.items():
        actual = metadata.version(package)
        installed[package] = actual
        assert actual.split('+')[0] == version, (package, actual, version)
    import numpy as np
    import torch, torchvision, transformers, timm, lightning, pytest
    from PIL import Image
    from pycocotools.coco import COCO
    from models.image_processing_deformable_detr import DeformableDetrImageProcessor
    from datasets.coco_hug import CocoDetection
    from autocheckout.taskcfg import TaskConfig
    import main as pdp_main
    torch.set_num_threads(1)
    torch.manual_seed(CONFIG['seed'])
    np.random.seed(CONFIG['seed'])
    evidence.update(packages=installed, cuda_available=torch.cuda.is_available(),
                    torch_cuda_runtime=torch.version.cuda,
                    gpu_name=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
    RUN['environment'] = {k:v for k,v in evidence.items() if k not in {'status','started_at','finished_at'}}
    print(json.dumps(evidence, indent=2))

## 3. Gate release/checksums và mapping

Đọc release hiện có, không sinh split mới. Kiểm tra source COCO, task config, split MD5 và toàn bộ task files theo manifest. `test_full.json` chỉ được hash ở đây, không dùng làm batch/train hoặc chọn threshold. Source/holdout được hash lại ở cuối.

In [ ]:
with gate('release_lock') as evidence:
    LOCK = load_json(REPO/'configs/data/rpc_100-4x25_seed0_native_v1.json')
    TASKS = RELEASE/'tasks/real'
    task_config_path = RELEASE/'task_config.json'
    task_manifest_path = TASKS/'manifest.json'
    assert hashlib.sha256(task_manifest_path.read_bytes()).hexdigest() == LOCK['task_manifest_sha256']
    assert hashlib.sha256(task_config_path.read_bytes()).hexdigest() == LOCK['input']['task_config_sha256']
    TASK_CONFIG = TaskConfig.load(task_config_path)
    assert [t.num_classes for t in TASK_CONFIG.data_tasks] == [100,25,25,25,25]
    assert (TASK_CONFIG.num_slots, TASK_CONFIG.num_classes) == (224,225)
    assert TASK_CONFIG.seed == 0 and LOCK['image_root'] == 'data/archive'
    assert not LOCK['synthetic_included'] and LOCK['coordinates'] == 'native_pixels'
    SOURCE_BEFORE = {}
    for file_name, digest in LOCK['input']['source_sha256'].items():
        path = RAW/file_name
        assert hashlib.sha256(path.read_bytes()).hexdigest() == digest, path
        SOURCE_BEFORE[str(path)] = ('sha256', digest)
    for file_name, digest in LOCK['split_md5'].items():
        path = RELEASE/'splits'/file_name
        assert md5_file(path) == digest, path
        SOURCE_BEFORE[str(path)] = ('md5', digest)
    TASK_MANIFEST = load_json(task_manifest_path)
    for file_name, details in TASK_MANIFEST['files'].items():
        path = TASKS/file_name
        assert md5_file(path) == details['md5'], path
        SOURCE_BEFORE[str(path)] = ('md5', details['md5'])
    assert md5_file(TASKS/'test_full.json') == LOCK['test_full_md5']
    SOURCE_BEFORE[str(task_config_path)] = ('sha256', LOCK['input']['task_config_sha256'])
    SOURCE_BEFORE[str(task_manifest_path)] = ('sha256', LOCK['task_manifest_sha256'])
    evidence.update(protocol=[100,25,25,25,25], slots=224, classifier_outputs=225,
                    task_config=str(task_config_path), checked_files=len(SOURCE_BEFORE))

## 4. Gate train labels/full GT — đủ 5 task

Train chỉ chứa current-task labels. Đối chiếu annotation IDs với `_gt_full` trên cùng ảnh để chứng minh filtering bảo toàn bbox/area và nhãn hiện tại. Full GT chỉ đọc cho audit ở cell này, không truyền vào loader train. Ảnh nhiều SKU có thể xuất hiện trong nhiều task; notebook ghi quy ước này, không chứng nhận mỗi ảnh chỉ được dùng một lần.

In [ ]:
with gate('task_gt_separation') as evidence:
    TASK_ROWS = []
    IMAGE_MEMBERSHIP = {}
    SELECTED_IDS = {}
    for task in TASK_CONFIG.data_tasks:
        current = load_json(TASKS/f'train_task_{task.task_id}.json')
        full = load_json(TASKS/f'train_task_{task.task_id}_gt_full.json')
        allowed = set(task.labels)
        assert {a['category_id'] for a in current['annotations']} == allowed
        assert {c['id'] for c in current['categories']} == allowed
        assert {i['id'] for i in current['images']} == {i['id'] for i in full['images']}
        assert current['images'] == full['images']
        full_by_id = {a['id']: a for a in full['annotations']}
        expected_ids = {a['id'] for a in full['annotations'] if a['category_id'] in allowed}
        assert len(full_by_id) == len(full['annotations'])
        assert {a['id'] for a in current['annotations']} == expected_ids
        assert len(expected_ids) == len(current['annotations'])
        for annotation in current['annotations']:
            assert annotation == full_by_id[annotation['id']]
        SELECTED_IDS[task.task_id] = sorted(i['id'] for i in current['images'])[:CONFIG['batch_size']]
        assert len(SELECTED_IDS[task.task_id]) == CONFIG['batch_size']
        for image in current['images']:
            IMAGE_MEMBERSHIP.setdefault(image['id'], []).append(task.task_id)
        TASK_ROWS.append({'task': task.task_id, 'labels': [min(allowed), max(allowed)],
                          'train_images': len(current['images']), 'train_objects': len(current['annotations']),
                          'full_gt_objects': len(full['annotations']),
                          'labels_removed': len(full['annotations'])-len(current['annotations']),
                          'selected_image_ids': SELECTED_IDS[task.task_id]})
        del current, full, full_by_id
    reused = sum(len(tasks)>1 for tasks in IMAGE_MEMBERSHIP.values())
    evidence.update(tasks=TASK_ROWS, images_used_in_multiple_tasks=reused,
                    protocol='current-task GT; cross-task image reuse permitted')
    print(json.dumps(evidence, indent=2))

## 5. Processor 640/800: geometry, normalization và postprocess

Với mỗi task/resolution: lấy đúng 2 ảnh train, kiểm tra image root/size, không đổi COCO index, labels/dtypes/mask; giải mã normalized cxcywh về pixel gốc và pixel đã resize. Sai số round-trip tối đa 0.02 pixel. Dùng chính postprocessor của repo với predictions tổng hợp để kiểm tra tọa độ về native pixels; đây không phải dự đoán model.

In [ ]:
from copy import deepcopy
from types import SimpleNamespace
from torch.utils.data import DataLoader, Subset, RandomSampler

IMAGE_SHA_BEFORE = {}
LOADERS = {}
PROCESSOR_RESULTS = []
for resolution in CONFIG['resolutions']:
    with gate(f'processor_{resolution}') as evidence:
        processor = DeformableDetrImageProcessor(size={'shortest_edge': resolution, 'longest_edge': resolution})
        assert processor.do_resize and processor.do_rescale and processor.do_normalize
        np.testing.assert_allclose(processor.image_mean, [0.485, 0.456, 0.406])
        np.testing.assert_allclose(processor.image_std, [0.229, 0.224, 0.225])
        rows = []
        for task in TASK_CONFIG.data_tasks:
            train_path = TASKS/f'train_task_{task.task_id}.json'
            assert not train_path.stem.endswith('gt_full')
            dataset = CocoDetection(str(RAW), str(train_path), processor, augment=None)
            indices = [dataset.ids.index(iid) for iid in SELECTED_IDS[task.task_id]]
            selected = Subset(dataset, indices)
            samples = []
            for idx in indices:
                iid = dataset.ids[idx]
                image_info = dataset.coco.loadImgs(iid)[0]
                image_path = (RAW/image_info['file_name']).resolve()
                assert image_path.is_relative_to(RAW) and image_path.is_file()
                digest = hashlib.sha256(image_path.read_bytes()).hexdigest()
                assert IMAGE_SHA_BEFORE.setdefault(str(image_path), digest) == digest
                with Image.open(image_path) as image:
                    width, height = image.size
                    assert (width,height) == (image_info['width'], image_info['height'])
                original_annotations = deepcopy(dataset.coco.imgToAnns[iid])
                pixels, target = dataset[idx]
                assert dataset.coco.imgToAnns[iid] == original_annotations
                valid = [a for a in original_annotations if not a.get('iscrowd',0)]
                original_boxes = torch.tensor([a['bbox'] for a in valid], dtype=torch.float32)
                expected_labels = torch.tensor([a['category_id'] for a in valid], dtype=torch.int64)
                assert torch.equal(target['class_labels'], expected_labels)
                assert set(expected_labels.tolist()) <= set(task.labels)
                assert torch.equal(target['orig_size'], torch.tensor([height,width]))
                assert pixels.shape[0] == 3 and pixels.dtype == torch.float32
                resized_h, resized_w = target['size'].tolist()
                assert max(resized_h,resized_w) == resolution
                assert pixels.shape[-2:] == (resized_h,resized_w)
                assert torch.isfinite(pixels).all() and torch.isfinite(target['boxes']).all()
                center, size = target['boxes'][:,:2], target['boxes'][:,2:]
                native_xywh = torch.cat([center-size/2,size], dim=1)*torch.tensor([width,height,width,height])
                torch.testing.assert_close(native_xywh, original_boxes, atol=0.02, rtol=0)
                resized_xywh = torch.cat([center-size/2,size], dim=1)*torch.tensor([resized_w,resized_h,resized_w,resized_h])
                torch.testing.assert_close(resized_xywh, original_boxes*torch.tensor([resized_w/width,resized_h/height]*2), atol=0.02, rtol=0)
                normalized = np.asarray(processor.image_mean)[:,None,None] + pixels.numpy()*np.asarray(processor.image_std)[:,None,None]
                assert normalized.min() >= -1e-5 and normalized.max() <= 1+1e-5
                # Recover all boxes through the repo postprocessor using synthetic confident logits.
                n = len(valid)
                logits = torch.full((1,n,225),-30.0)
                logits[0,torch.arange(n),expected_labels] = 30.0
                output = SimpleNamespace(logits=logits, pred_boxes=target['boxes'].unsqueeze(0))
                recovered = processor.post_process_object_detection(output, threshold=0.9,
                    target_sizes=target['orig_size'].unsqueeze(0), top_k=n)[0]
                expected_xyxy = torch.cat([original_boxes[:,:2],original_boxes[:,:2]+original_boxes[:,2:]],dim=1)
                assert len(recovered['boxes']) == n
                # Equal scores can reorder detections; match each returned box+class to its source.
                unused = list(range(n))
                for box, label in zip(recovered['boxes'], recovered['labels']):
                    matches = [j for j in unused if int(expected_labels[j])==int(label)
                               and torch.allclose(box,expected_xyxy[j],atol=0.02,rtol=0)]
                    assert matches, (iid, label, box)
                    unused.remove(matches[0])
                assert not unused
                rows.append({'task':task.task_id,'image_id':iid,'native':[height,width],
                             'processed':[resized_h,resized_w],'objects':n})
                samples.append((pixels,target))
            batch = dataset.collate_fn(samples)
            assert len(batch['labels']) == CONFIG['batch_size']
            assert batch['pixel_mask'].shape == batch['pixel_values'].shape[:1]+batch['pixel_values'].shape[2:]
            assert set(batch['pixel_mask'].unique().tolist()) <= {0,1}
            for bidx, target in enumerate(batch['labels']):
                h,w = target['size'].tolist()
                expected_mask = torch.zeros_like(batch['pixel_mask'][bidx]); expected_mask[:h,:w] = 1
                assert torch.equal(batch['pixel_mask'][bidx],expected_mask)
            # The wrapper preserves CocoDetection.collate_fn for main.make_train_loader.
            selected.collate_fn = dataset.collate_fn
            LOADERS[resolution,task.task_id] = selected
        evidence.update(samples=rows, max_native_box_error_pixels=0.02,
                        size_policy='aspect-preserving shortest/longest edge cap + batch padding')
        PROCESSOR_RESULTS.extend(rows)
        print(resolution, ':', len(rows), 'real images checked')
        gc.collect()

## 6. Loader/collation và train shuffle

Gọi chính `pdp.main.make_train_loader` với subset đã khóa, current-task JSON và workers=0. Kiểm tra random sampler, batch IDs/labels/mask và seed tái lập. Kiểm tra toàn bộ train labels nằm ở cell 4; cell này chỉ lấy mẫu ảnh thật. `Subset` chỉ giới hạn lượng dữ liệu nghiệm thu, không đổi split.

In [ ]:
for resolution in CONFIG['resolutions']:
    with gate(f'loader_{resolution}') as evidence:
        checked = []
        for task in TASK_CONFIG.data_tasks:
            selected = LOADERS[resolution,task.task_id]
            loader_args = SimpleNamespace(batch_size=CONFIG['batch_size'], num_workers=0, shuffle=1)
            torch.manual_seed(CONFIG['seed'])
            loader = pdp_main.make_train_loader(selected, loader_args)
            assert isinstance(loader.sampler, RandomSampler)
            torch.manual_seed(CONFIG['seed'])
            first = next(iter(loader))
            torch.manual_seed(CONFIG['seed'])
            second = next(iter(pdp_main.make_train_loader(selected, loader_args)))
            ids = [int(t['image_id'].item()) for t in first['labels']]
            assert ids == [int(t['image_id'].item()) for t in second['labels']]
            assert set(ids) == set(SELECTED_IDS[task.task_id])
            assert len(ids) == CONFIG['batch_size']
            for target in first['labels']:
                assert set(target['class_labels'].tolist()) <= set(task.labels)
            assert torch.isfinite(first['pixel_values']).all()
            checked.append({'task':task.task_id,'ids':ids,'batch_shape':list(first['pixel_values'].shape)})
        evidence.update(tasks=checked, workers=0, shuffled=True, deterministic_same_seed=True)

## 7. PDP nhỏ: forward/backward, losses và optimizer step

Dùng factory nhỏ từ `tests/pdp_helpers.py` trong phạm vi `pytest.MonkeyPatch.context()`, rồi khôi phục ngay. Dùng **mapping thật** của release và 2 ảnh thật Task 1 đã chọn, processor 96. Bật shared/private, DDL=0.15, query loss gradient, giữ prototype mean; không bật FSA/freeze shared bổ sung/LoRA/K=3.

Kiểm tra loss và gradient finite, gradient khác 0 vào classifier/query_tf, optimizer chứa đúng params trainable, một bước làm đổi params học; backbone/encoder/decoder frozen giữ hash. AdamW step nhỏ này là smoke, không phải lượt train baseline hay convergence.

In [ ]:
if not CONFIG['run_smoke']:
    skip('pdp_smoke')
else:
    with gate('pdp_smoke') as evidence:
        from pdp_helpers import use_tiny_detr, main_args, make_trainer, TINY_DETR
        work = ARTIFACT/'pdp_smoke'; work.mkdir()
        torch.manual_seed(CONFIG['seed'])
        args = main_args(['--task_config',str(task_config_path),'--n_classes','225',
            '--repo_name','','--n_tasks','5','--use_prompts','1','--local_query','1',
            '--num_prompts','4','--prompt_len','2','--lambda_query','0.1','--ddl_lambda','0.15',
            '--query_loss_grad','1','--use_shared','1','--use_private','1',
            '--freeze','backbone,encoder,decoder','--freeze_shared_after_task1','0',
            '--new_params','class_embed,prompts','--optim_groups','prompt',
            '--accelerator','cpu','--batch_size','2','--num_workers','0',
            '--epochs','2','--output_dir',str(work)])
        pdp_main.setup_task_info(args)
        with (work/'model.log').open('w') as model_log, pytest.MonkeyPatch.context() as patch:
            args.log_file = model_log
            use_tiny_detr(patch)
            trainer = make_trainer(args, task_id=1)
            trainer.model.model.prompts.init_task_prompts()
            processor = DeformableDetrImageProcessor(size={'shortest_edge':96,'longest_edge':96})
            trainer.processor = processor
            dataset = CocoDetection(str(RAW),str(TASKS/'train_task_1.json'),processor)
            batch = dataset.collate_fn([dataset[dataset.ids.index(iid)] for iid in SELECTED_IDS[1]])
            assert trainer.model.config.num_labels == 225 and trainer.seen_classes == 100
            assert trainer.args.optim_groups == 'prompt'
            assert trainer.invalid_cls_logits == list(range(100,224))
            optimizers = trainer.configure_optimizers()
            optimizer = optimizers['optimizer']
            trained_ids = [id(p) for group in optimizer.param_groups for p in group['params']]
            expected_ids = {id(p) for p in trainer.parameters() if p.requires_grad}
            assert len(trained_ids)==len(set(trained_ids)) and set(trained_ids)==expected_ids
            names = dict(trainer.named_parameters())
            def frozen_digest():
                digest = hashlib.sha256()
                for name,p in names.items():
                    if not p.requires_grad:
                        digest.update(name.encode()); digest.update(p.detach().cpu().contiguous().numpy().tobytes())
                return digest.hexdigest()
            frozen_before = frozen_digest()
            classifier = next(p for n,p in names.items() if 'class_embed' in n and n.endswith('weight') and p.requires_grad)
            classifier_before = classifier.detach().clone()
            trainer.train(); optimizer.zero_grad(set_to_none=True)
            loss, losses = trainer.common_step(batch,0)
            assert {'loss_ce','loss_bbox','loss_giou','loss_ddl','query_loss'} <= set(losses)
            assert loss.requires_grad and torch.isfinite(loss)
            assert all(torch.isfinite(value).all() for value in losses.values())
            loss.backward()
            grads = {n:float(p.grad.norm()) for n,p in names.items() if p.requires_grad and p.grad is not None}
            assert grads and all(torch.isfinite(p.grad).all() for p in names.values() if p.grad is not None)
            assert any(v>0 for n,v in grads.items() if 'class_embed' in n)
            assert any(v>0 for n,v in grads.items() if 'query_tf' in n)
            grad_norm = torch.nn.utils.clip_grad_norm_([p for p in trainer.parameters() if p.requires_grad],0.1)
            assert torch.isfinite(grad_norm)
            optimizer.step()
            assert optimizer.state and not torch.equal(classifier_before,classifier.detach())
            assert frozen_digest()==frozen_before
            evidence.update(total_loss=float(loss.detach()), losses={k:float(v.detach()) for k,v in losses.items()},
                gradient_norm=float(grad_norm), nonzero_gradient_tensors=sum(v>0 for v in grads.values()),
                optimizer_groups=[{'lr':g['lr'],'parameters':sum(p.numel() for p in g['params'])} for g in optimizer.param_groups],
                frozen_sha256=frozen_before, updated_classifier=True, optimizer_steps=1,
                model_profile=TINY_DETR, resolution=96, pretrained=False, real_image_ids=SELECTED_IDS[1])
            save_json(work/'args.json',{k:v for k,v in vars(args).items() if k!='log_file'},indent=2)
            print(json.dumps(evidence,indent=2))
            del loss, losses, optimizer, optimizers, classifier, classifier_before, names, trainer, grads
        gc.collect()

## 8. Regression PDP nền và resume thật của Lightning

Chạy tests sẵn có bằng đúng interpreter của kernel, tách subprocess để giải phóng model smoke. Log/JUnit/tmp fixtures nằm trong run directory. Regression kiểm tra pools, DDL/query gradient, teacher, PPG/prototypes, shuffle và task transition. Resume test cố tình ngắt epoch rồi khôi phục `last.ckpt`, optimizer/scheduler/global step/prototype cache qua đường Lightning/main hiện có. Không coi `task_final.pth` là checkpoint resume đầy đủ; test cũng ghi giới hạn thứ tự dữ liệu sau resume.

Các tests dùng fixture nhỏ, không dùng real holdout, không tải pretrained weights. Nếu skip/fail/error, gate không PASS. Có thể mất vài phút trên CPU.

In [ ]:
import xml.etree.ElementTree as ET

def run_regression(name, paths):
    with gate(name) as evidence:
        junit = ARTIFACT/f'{name}.xml'
        log = ARTIFACT/f'{name}.log'
        command = [sys.executable,'-m','pytest',*paths,'-q','-o','addopts=',
                   '--tb=short','--junitxml',str(junit),'--basetemp',str(ARTIFACT/f'{name}_tmp')]
        environment = os.environ.copy()
        environment.update(USE_TF='0', HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1',
                           OMP_NUM_THREADS='1',MKL_NUM_THREADS='1',MPLBACKEND='Agg')
        evidence.update(command=command,log=str(log),junit=str(junit),python=sys.executable)
        persist()
        with log.open('w') as output:
            result = subprocess.run(command,cwd=REPO,env=environment,stdout=output,
                                    stderr=subprocess.STDOUT,timeout=1800)
        evidence['exit_code'] = result.returncode
        print('\n'.join(log.read_text().splitlines()[-25:]))
        assert result.returncode==0, f'Regression failed: see {log}'
        suites = list(ET.parse(junit).getroot().iter('testsuite'))
        counts = {key:sum(int(s.get(key,0)) for s in suites) for key in ['tests','errors','failures','skipped']}
        evidence['counts'] = counts
        assert counts['tests']>0 and counts['errors']==counts['failures']==counts['skipped']==0, counts

if CONFIG['run_baseline_regression']:
    run_regression('baseline_regression',[
        'tests/test_pdp_f1_task_config.py','tests/test_pdp_f2_private_pool.py',
        'tests/test_pdp_f3_ddl.py','tests/test_pdp_f4_query_loss.py',
        'tests/test_pdp_f5_teacher.py','tests/test_pdp_f6_ppg.py',
        'tests/test_pdp_f7_prototypes.py','tests/test_pdp_f12_shuffle.py',
        'tests/test_pdp_f8_paths.py'])
else:
    skip('baseline_regression')

In [ ]:
if CONFIG['run_resume_regression']:
    run_regression('lightning_resume',['tests/test_pdp_r1_resume.py'])
else:
    skip('lightning_resume')

## 9. Source preservation và kết thúc

Hash lại source COCO, task/split JSON, holdout và các ảnh đã đọc. Chỉ công nhận `PASS_LIMITED_PDP_FOUNDATION` khi mọi gate bắt buộc PASS; thiếu/tắt gate cho PARTIAL. Trạng thái này vẫn chưa nghiệm thu full pretrained PDP/GPU, đường CLI processor 640/800 trong training, ConvNeXt hoặc hiệu quả học.

Nếu kernel bị kill mà chưa có finished_at, giữ trạng thái RUNNING như run dang dở, không tự đổi thành PASS. Lưu notebook với outputs sau khi chạy; có thể Save As thêm bản vào `runs/notebook_versions/01_processor_loader_pdp_acceptance/<run_id>.ipynb`.

In [ ]:
with gate('sources_preserved') as evidence:
    for path,(algorithm,expected) in SOURCE_BEFORE.items():
        actual = md5_file(path) if algorithm=='md5' else hashlib.sha256(Path(path).read_bytes()).hexdigest()
        assert actual==expected, f'Locked source/release changed: {path}'
    for path,expected in IMAGE_SHA_BEFORE.items():
        assert hashlib.sha256(Path(path).read_bytes()).hexdigest()==expected, path
    evidence.update(locked_files=len(SOURCE_BEFORE),selected_images=len(IMAGE_SHA_BEFORE),
                    source_unchanged=True,holdout_unchanged=True)

missing = [name for name in REQUIRED if RUN['gates'].get(name,{}).get('status')!='PASS']
RUN['finished_at'] = now()
RUN['duration_seconds'] = (datetime.fromisoformat(RUN['finished_at'])-datetime.fromisoformat(RUN['started_at'])).total_seconds()
failed = [name for name in REQUIRED if RUN['gates'].get(name,{}).get('status') in {'FAIL','INTERRUPTED'}]
RUN['status'] = 'FAIL' if failed else ('PARTIAL' if missing else 'PASS_LIMITED_PDP_FOUNDATION')
RUN['missing_or_failed_gates'] = missing
RUN['limitations'] = [
    'PDP smoke uses a tiny random ResNet18 at 96 pixels, not pretrained full ResNet50.',
    'Real processor/loader checked at 640/800; full-model forward/backward at those sizes remains pending.',
    'Cross-task image reuse is allowed by this locked research split.',
    'No ConvNeXt adapter, GPU/kernel acceptance, benchmark or convergence result.',
    'Synthetic augmentation and calibration are not integrated by this notebook.']
persist()
save_json(ARTIFACT/'summary.json',RUN,indent=2)
save_json(ARTIFACT.parent/'latest.json',{'run_id':RUN['run_id'],'status':RUN['status'],
          'summary':str(ARTIFACT/'summary.json'),'finished_at':RUN['finished_at']},indent=2)
import shutil
source_notebook = REPO/'notebooks/modeling/01_processor_loader_pdp_acceptance.ipynb'
shutil.copy2(source_notebook, ARTIFACT/'notebook_source.ipynb')
print(json.dumps(RUN,indent=2,ensure_ascii=False))
print('Save this notebook after execution to retain cell outputs.')